# Erzeuge Embeddings für text Chunks
Die Chunks müssen vorher erstellt und in der Tabelle bge_m3_vectors gespeichert werden

## Environment

In [1]:
import os
os.environ["IND_PG_SCHEMA"] = "meipi-indexing"
os.environ["IND_DATA_DIR"] = "/home/padmin/Development/projekte/meipi-indexing/data"

## Import

In [2]:
import sqlalchemy as sa
from sqlalchemy.orm import aliased
from meipi.indexing import DBOperations, DBBgeM3Vector
from meipi.indexing.model import ChunkItem
from meipi.indexing.embedding import EmbeddingConfig, EmbeddingPipeline


## Konfiguration

In [3]:
pool_id = 1
qsize = 100
num_workers = 4
emb_config = EmbeddingConfig(max_queue_size=qsize, num_workers=num_workers)


## Lies Chunks von DB

In [6]:
numchunks = 100
V = aliased(DBBgeM3Vector)
dbop = DBOperations(pool_id)
with dbop.Session() as session:
    res = session.execute(sa.select(V.doc_id, V.chunk_index, V.content).limit(numchunks)).fetchall()
    chunklist = [ChunkItem(**row._asdict()) for row in res]
print(len(chunklist))
print(chunklist[0])

100
ChunkItem(doc_id=112990, chunk_index=76, content='maler, die Brauen geschwungener. Nur das Lächeln war gleich, eindringlich und irgendwie... gefährlich. „Ich sehe besser aus", sagte Rafe augenzwinkernd und kam die Treppe herab. „Die Ähnlichkeit ist erstaunlich." Savannah streckte die Hand aus. „Sie müssen Rafe MacKade sein." „Schuldig." „Ich bin..." „Savannah Morningstar." Er schüttelte ihre Hand nicht, sondern hielt sie fest, während er Savannah gründlich musterte. „Regan hatte vollkommen recht." „Wie bitte?" „Sie waren letzte Woche im Laden meiner Frau. Regan hat Sie mir beschrieben und meinte, ich solle Sie mir wie Isis, die ägyptische Göttin, vorstellen. Damit konnte ich nicht viel anfangen, ehrlich gesagt. Also meinte sie, ich solle mir eine Frau vorstellen, die einem Mann den Atem raubt." „Das ist ein ziemlich gewagtes Kompliment." „Und eins, das zutrifft", sagte er. „Jared hat mir erzählt, dass Sie vorbeikommen würden." Er hakte die Daumen hinter den Werkzeuggürtel. „Ich möc

## Run Pipeline

In [7]:
pipeline = EmbeddingPipeline(emb_config,pool_id)
pipeline.run_pipeline(chunklist)


Starting ingest process at 1781881115.561858


Ingesting chunks: 100%|██████████| 100/100 [00:00<00:00, 161133.46it/s]
Tokenizing chunks: 15it [00:00, 586.35it/s]/s]
Tokenizing chunks: 19it [00:00, 711.90it/s]
Tokenizing chunks: 0it [00:00, ?it/s]
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 29939.62it/s]
Tokenizing chunks: 66it [00:00, 70.56it/s]
Writing chunks to database: 100it [00:05, 17.52it/s]


End time: 1781881131.6687691 Start time: 1781881115.561858
Pipeline finished in 16.106911182403564 seconds


## Test

In [ ]:
from sqlalchemy.sql import null


with dbop.Session() as session:
    stmt = sa.select(sa.func.count()).select_from(DBBgeM3Vector).where(DBBgeM3Vector.content == null())
    res = session.execute(stmt).scalar_one()
    print(res)


253974
